In [30]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm import tqdm

from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
from functions_hmm import *


plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

qt5agg
qt5agg


# Simulations generations

## Parameters

In [31]:
n_simulations = 50

n_simulations_list = [n_simulations]*100
p_cw_init_range = np.linspace(0,1,100)

start_index = 0

# simulations_folder_path = f'/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift'
simulations_folder_path = f'/home/david/Documents/code/DDM_vXbis_synthetic_data_identical_drift_0050'
simulations_folder_path = '/home/tom/Code/ddm_simulations/DDM_vXbis_synthetic_data_identical_drift_0050'

# simulations_folder_path = f'/home/tom/Code/ddm_simulations/simulations_batches_bis/'


# Drift Estimation

## Generating "theoretical" average probability sequences

In [32]:
generate_theoretical_sequences = False # PARAM

if generate_theoretical_sequences:

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index +1}.pkl', 'rb') as file:
                synthetic_data = pickle.load(file)

    args = [synthetic_data[0]['parameters']] + [5000] + [p_cw_init_range]
    drift_range = np.linspace(0.01,0.2,200)

    # test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts(drift_range, args)
    test_average_probability_sequences = generate_test_average_probability_sequences_identical_drifts_random_init(drift_range, args)

    with open(f'{simulations_folder_path}/test_average_probability_sequences.pkl', 'wb') as file:
        pickle.dump([drift_range, test_average_probability_sequences], file)

else:
 
    with open(f'{simulations_folder_path}/test_average_probability_sequences.pkl', 'rb') as file:
        drift_range, test_average_probability_sequences = pickle.load(file)


## Minimizing mean square error

In [33]:
save_result = True # PARAM

# for index, _ in enumerate(tqdm(n_simulations_list)):
for index, _ in enumerate(n_simulations_list):

    average_proba_sequence_hmm = []

    ##############################
    ### Loading Data and Model ###
    ##############################

    with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        synthetic_data = pickle.load(file)

    with open(f'{simulations_folder_path}/n_{n_simulations}/model_score_fit_output_{n_simulations}_test_{start_index + index+1}.pkl', 'rb') as file:
        fit_output = pickle.load(file)

    # model = fit_output['models'][np.argmax(fit_output['scores'])]
    model = fit_output['models'][8]

    ########################
    ### Reformating Data ###
    ########################

    test_data = [synth_data['choices'] for synth_data in synthetic_data]

    emissionmat = model.emissionprob_
    transmat = model.transmat_
    initial_state_distri = model.startprob_ # compute_initial_state_list_distri(model,test_data)

    ####################
    ### Computations ###
    ####################

    #######

    """    mat_i = copy.deepcopy(transmat)

    for i in range(len(test_data[0])):

        mat_i = np.matmul(mat_i,transmat)
        res = np.matmul(initial_state_distri,mat_i)*emissionmat[:,1]
    
        average_proba_sequence_hmm.append(np.sum(res))
    """
    #######

    average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(test_data, model)

    # average_proba_sequences_hmm.append(average_proba_sequence_hmm)

    #######

    mse_list = []

    # for test_msd_sequence in tqdm(test_msd_sequence_list, leave=False):
    for test_average_probability_sequence in test_average_probability_sequences:
    
        mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(mse_list)
    recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]

    if not(save_result):

        continue

    ##############################
    ### Saving Recovered drift ###
    ##############################
    
    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'wb') as file:
        pickle.dump([test_average_probability_sequences,recovered_drift], file)


In [34]:
recovered_drift_list = []

for index, _ in enumerate(n_simulations_list):

    with open(f'{simulations_folder_path}/n_{n_simulations}/recovered_drift_{n_simulations}_{start_index + index+1}.pkl', 'rb') as file:
        _, recovered_drift = pickle.load(file)
    
    recovered_drift_list.append(recovered_drift[0])

# Plots

In [37]:
example_set = 5

average_proba_sequence_hmm = []

##############################
### Loading Data and Model ###
##############################

with open(f'{simulations_folder_path}/n_{n_simulations}/simulations_batch_{n_simulations}_test_{example_set}.pkl', 'rb') as file:
    synthetic_data = pickle.load(file)

with open(f'{simulations_folder_path}/n_{n_simulations}/model_score_fit_output_{n_simulations}_test_{example_set}.pkl', 'rb') as file:
    fit_output = pickle.load(file)

model = fit_output['models'][np.argmax(fit_output['scores'])]

test_data = [synth_data['choices'] for synth_data in synthetic_data]

average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(test_data, model)

mse_list = []

# for test_msd_sequence in tqdm(test_msd_sequence_list, leave=False):
for test_average_probability_sequence in test_average_probability_sequences:
   
    mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

min_mse = np.min(mse_list)
recovered_drift = drift_range[np.where(mse_list==min_mse)[0]]

index_min_mse = np.where(mse_list==min_mse)[0][0]

average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(test_data, model)

####################
### Computations ###
####################

fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

ax1.plot(average_proba_sequence_hmm)
ax1.plot(test_average_probability_sequences[index_min_mse])
ax1.text(20, 0.5, f'drift = {np.round(recovered_drift[0],4)}')

ax1.set_ylim([0,1])
ax1.set_xlabel('Step')
ax1.set_ylabel('Average proba')

ax1.legend()

/tmp/ipykernel_18712/2815055884.py:53: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax1.legend()


In [36]:
fig=plt.figure(figsize=(4, 4), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])

histo = np.histogram(recovered_drift_list, bins=np.linspace(0.01,0.2,51))
bin_width = histo[1][1] - histo[1][0]
ax1.stairs(histo[0], histo[1], alpha=0.5, fill=True, label = 'Recovered drift probability density')

ax1.axvline(0.050, linewidth=0.7, color='k', linestyle='--', label='Drift to recover')

# ax1.set_xticks([])
ax1.set_title(f'{len(n_simulations_list)} sets of {n_simulations} simulations')

ax1.set_ylim([0,None])
ax1.set_xlabel('Recovered drift')
ax1.set_ylabel('Probability density')

ax1.legend()